In [1]:
import hashlib
import time

# -----------------------------
# Simulated stored user data
# -----------------------------
stored_username = "admin"

# Hashing the password (simulating database storage)
stored_password_hash = hashlib.sha256("secure@123".encode()).hexdigest()

MAX_ATTEMPTS = 3
attempts = 0
lock_time = 10  # seconds (temporary lock)


def hash_password(password):
    """Converts plain password to hashed form"""
    return hashlib.sha256(password.encode()).hexdigest()


def authenticate(username, password):
    """Validates user credentials"""
    return (
        username == stored_username and
        hash_password(password) == stored_password_hash
    )


# -----------------------------
# Login Logic
# -----------------------------
while attempts < MAX_ATTEMPTS:
    username = input("Enter username: ").strip()
    password = input("Enter password: ").strip()

    if authenticate(username, password):
        print("✅ Login successful. Welcome to the application.")
        break
    else:
        attempts += 1
        print(f"❌ Invalid credentials. Attempts left: {MAX_ATTEMPTS - attempts}")

    if attempts == MAX_ATTEMPTS:
        print("🔒 Account locked temporarily due to multiple failed attempts.")
        print(f"⏳ Please wait {lock_time} seconds before trying again.")
        time.sleep(lock_time)
        attempts = 0   # reset attempts after lock period


❌ Invalid credentials. Attempts left: 2
❌ Invalid credentials. Attempts left: 1
✅ Login successful. Welcome to the application.


In [ ]:
import hashlib
import time

class SecureLoginSystem:
    def __init__(self):
        # Simulated user database: username -> (hashed_password, is_locked, lock_until)
        self.users = {
            'user1': (self.hash_password('password123'), False, 0),
            'admin': (self.hash_password('adminpass'), False, 0)
        }
        self.max_attempts = 3
        self.lock_duration = 60  # seconds (1 minute for demo)

    def hash_password(self, password):
        """Hash password using SHA-256 for security."""
        return hashlib.sha256(password.encode('utf-8')).hexdigest()

    def validate_input(self, input_str):
        """Validate input: strip whitespace and check if not empty."""
        cleaned = input_str.strip()
        if not cleaned:
            raise ValueError("Input cannot be empty.")
        return cleaned

    def is_locked(self, username):
        """Check if account is locked and if lock has expired."""
        if username not in self.users:
            return False
        _, is_locked, lock_until = self.users[username]
        if is_locked and time.time() < lock_until:
            return True
        if is_locked:  # Unlock if time expired
            self.users[username] = (self.users[username][0], False, 0)
        return False

    def lock_account(self, username):
        """Lock the account temporarily."""
        if username in self.users:
            lock_until = time.time() + self.lock_duration
            self.users[username] = (self.users[username][0], True, lock_until)
            print(f"Account '{username}' locked for {self.lock_duration} seconds.")

    def login(self):
        """Main login logic with attempt counter and validation."""
        attempts = 0
        while attempts < self.max_attempts:
            try:
                username = self.validate_input(input("Enter username: "))
                if self.is_locked(username):
                    print("Account is temporarily locked. Try again later.")
                    return False

                password = self.validate_input(input("Enter password: "))
                hashed_pw = self.hash_password(password)

                if username in self.users and self.users[username][0] == hashed_pw:
                    print("Login successful!")
                    return True
                else:
                    attempts += 1
                    remaining = self.max_attempts - attempts
                    print(f"Invalid username or password. {remaining} attempts remaining.")
            except ValueError as e:
                print(f"Validation error: {e}")
                continue  # Don't count invalid input as an attempt

        # All attempts failed
        self.lock_account(username)
        print("Maximum attempts reached. Account locked.")
        return False

# Usage example
if __name__ == "__main__":
    system = SecureLoginSystem()
    system.login()


Invalid username or password. 2 attempts remaining.
Login successful!


In [ ]:
# login_system.py
import hashlib
import time
from getpass import getpass  # For secure password input (hides typing)


class SecureLoginSystem:
    def __init__(self, max_attempts=3, lock_duration=60):
        # Simulated user database: username -> (hashed_password, is_locked, lock_until_timestamp)
        self.users = {
            'user1': (self._hash_password('password123'), False, 0),
            'admin': (self._hash_password('secretadmin'), False, 0)
        }
        self.max_attempts = max_attempts
        self.lock_duration = lock_duration  # in seconds

    def _hash_password(self, password: str) -> str:
        """Hash password using SHA-256 (internal method)."""
        return hashlib.sha256(password.encode('utf-8')).hexdigest()

    def validate_input(self, input_str: str) -> str:
        """Validate and clean input: strip whitespace and reject empty."""
        if not input_str or not input_str.strip():
            raise ValueError("Input cannot be empty or consist only of whitespace.")
        return input_str.strip()

    def is_account_locked(self, username: str) -> bool:
        """Check if the account is currently locked. Auto-unlock if time expired."""
        if username not in self.users:
            return False
        _, locked, lock_until = self.users[username]
        if locked and time.time() < lock_until:
            return True
        if locked:  # Lock time has expired → unlock automatically
            self.users[username] = (self.users[username][0], False, 0)
        return False

    def lock_account(self, username: str):
        """Lock the account for the defined duration."""
        if username in self.users:
            lock_until = time.time() + self.lock_duration
            self.users[username] = (self.users[username][0], True, lock_until)

    def authenticate(self, username: str, password: str) -> bool:
        """Check if username exists and password matches the stored hash."""
        if username not in self.users:
            return False
        stored_hash, _, _ = self.users[username]
        return stored_hash == self._hash_password(password)

    def attempt_login(self, username: str, password: str) -> dict:
        """
        Simulate a single login attempt.
        Returns a dictionary with result details (highly testable).
        """
        try:
            self.validate_input(username)
            self.validate_input(password)
        except ValueError as e:
            return {"success": False, "message": str(e), "locked": False}

        if self.is_account_locked(username):
            return {"success": False, "message": "Account is temporarily locked.", "locked": True}

        if self.authenticate(username, password):
            return {"success": True, "message": "Login successful!", "locked": False}

        return {"success": False, "message": "Invalid username or password.", "locked": False}

    def run_interactive_login(self):
        """Interactive console-based login with attempt limiting and locking."""
        attempts = 0
        username = ""

        print("=== Secure Login System ===\n")

        while attempts < self.max_attempts:
            try:
                username = self.validate_input(input("Enter username: "))

                if self.is_account_locked(username):
                    print("Account is temporarily locked. Try again later.")
                    return False

                # Secure password input (characters not shown)
                password = getpass("Enter password: ")
                password = self.validate_input(password)

                result = self.attempt_login(username, password)

                if result["success"]:
                    print(f"\n{result['message']}")
                    return True

                # Failed attempt
                attempts += 1
                remaining = self.max_attempts - attempts
                attempt_word = "attempt" if remaining == 1 else "attempts"
                print(f"{result['message']} {remaining} {attempt_word} remaining.\n")

            except ValueError as e:
                print(f"Error: {e}\n")

        # Max attempts reached
        self.lock_account(username)
        print(f"\nToo many failed attempts.")
        print(f"Account '{username}' has been locked for {self.lock_duration} seconds.")
        return False


# Run the system when executed directly
if __name__ == "__main__":
    system = SecureLoginSystem(max_attempts=3, lock_duration=30)  # 30 seconds lock for demo
    system.run_interactive_login()


=== Secure Login System ===


Login successful!


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------------------------------
# Step 1: Mock User Activity Data
# -------------------------------------------------

data = {
    "user_id": [1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5],
    "product_id": [101, 102, 103, 101, 104, 102, 103, 105, 101, 105, 104],
    "action": [
        "view", "purchase", "click",
        "purchase", "view",
        "click", "purchase", "view",
        "click", "purchase", "view"
    ]
}

df = pd.DataFrame(data)

# -------------------------------------------------
# Step 2: Action to Score Mapping
# -------------------------------------------------

action_weight = {
    "view": 1,
    "click": 2,
    "purchase": 5
}

df["score"] = df["action"].map(action_weight)

# -------------------------------------------------
# Step 3: User–Item Interaction Matrix
# -------------------------------------------------

user_item_matrix = df.pivot_table(
    index="user_id",
    columns="product_id",
    values="score",
    aggfunc="sum",
    fill_value=0
)

# -------------------------------------------------
# Step 4: Collaborative Filtering (Item-Based)
# -------------------------------------------------

item_similarity = cosine_similarity(user_item_matrix.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

# -------------------------------------------------
# Step 5: Collaborative Recommendation Function
# -------------------------------------------------

def recommend_products(user_id, top_n=3):
    if user_id not in user_item_matrix.index:
        return "New user – use content-based or trending products"

    user_scores = user_item_matrix.loc[user_id]
    interacted_products = user_scores[user_scores > 0].index

    scores = pd.Series(dtype=float)

    for product in interacted_products:
        scores = scores.add(item_similarity_df[product], fill_value=0)

    scores = scores.drop(interacted_products, errors="ignore")
    return scores.sort_values(ascending=False).head(top_n)

# -------------------------------------------------
# Step 6: Content-Based Filtering
# -------------------------------------------------

products = pd.DataFrame({
    "product_id": [101, 102, 103, 104, 105],
    "description": [
        "running sports shoes",
        "casual leather shoes",
        "sports fitness shoes",
        "wireless bluetooth headphones",
        "noise cancelling over ear headphones"
    ]
})

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(products["description"])

product_similarity = cosine_similarity(tfidf_matrix)

# -------------------------------------------------
# Step 7: Content-Based Recommendation Function
# -------------------------------------------------

def recommend_similar_products(product_id, top_n=2):
    if product_id not in products["product_id"].values:
        return []

    idx = products[products["product_id"] == product_id].index[0]
    similarity_scores = list(enumerate(product_similarity[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similar_products = similarity_scores[1:top_n + 1]
    return [products.iloc[i[0]]["product_id"] for i in similar_products]

# -------------------------------------------------
# Step 8: Hybrid Recommendation
# -------------------------------------------------

def hybrid_recommendation(user_id):
    if user_id not in user_item_matrix.index:
        return "New user – show trending products"

    collab_recs = recommend_products(user_id)

    if isinstance(collab_recs, str):
        return collab_recs

    return list(collab_recs.index)
# -------------------------------------------------
# Step 9: Show Output When Run Directly
# -------------------------------------------------

if __name__ == "__main__":

    print("\n--- USER ACTIVITY DATA ---")
    print(df)

    print("\n--- USER-ITEM MATRIX ---")
    print(user_item_matrix)

    print("\n--- COLLABORATIVE RECOMMENDATIONS ---")
    for user in user_item_matrix.index:
        print(f"User {user} recommendations:")
        print(recommend_products(user))
        print("-" * 40)

    print("\n--- CONTENT-BASED RECOMMENDATIONS ---")
    for pid in products["product_id"]:
        print(f"Products similar to {pid}: {recommend_similar_products(pid)}")

    print("\n--- HYBRID RECOMMENDATION EXAMPLE ---")
    print("User 3:", hybrid_recommendation(3))
    print("New User:", hybrid_recommendation(999))




--- USER ACTIVITY DATA ---
    user_id  product_id    action  score
0         1         101      view      1
1         1         102  purchase      5
2         1         103     click      2
3         2         101  purchase      5
4         2         104      view      1
5         3         102     click      2
6         3         103  purchase      5
7         3         105      view      1
8         4         101     click      2
9         4         105  purchase      5
10        5         104      view      1

--- USER-ITEM MATRIX ---
product_id  101  102  103  104  105
user_id                            
1             1    5    2    0    0
2             5    0    0    1    0
3             0    2    5    0    1
4             2    0    0    0    5
5             0    0    0    1    0

--- COLLABORATIVE RECOMMENDATIONS ---
User 1 recommendations:
product_id
104    0.645497
105    0.612982
dtype: float64
----------------------------------------
User 2 recommendations:
product_id
105  

In [ ]:
pip install river


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 3.9 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 6.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 4.0 MB/s  0:00:05m0:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.13.1
    Uninstalling scipy-1.13.1:
      Successfully uninstalled scipy-1.13.1━━━━━ 0/3 [scipy]
  Attempting uninstall: pandas━━━━━━━━━━━━━━━━━━ 0/3 [scipy]
    Found existing installation: pandas 2.2.2 0/3 [scipy]
    Uninstalling pandas-2.2.2:╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [pandas]
      Successfully uninstalled pandas-2.2.2━━━━━━━━━━━━━━━━━━━ 1/3 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [river]32m2/3 [river]]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aext-panels 4.1.0 requires anaconda-cloud-auth>=0.7.1, whic